# HST capacity-feasible internal cough + speech fusion

This notebook operates a separate 20-job HST-Base profile: ten frozen participant-level repetitions for cough and ten for speech, followed by validation-frozen fusion. It does not authorize HST temporal or external-transfer claims.

In [ ]:
from pathlib import Path
import json, subprocess, sys, time

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
CONFIG = PROJECT_ROOT / 'configs' / 'hst_capacity_internal_fusion.json'
ACCEPTED = PROJECT_ROOT / 'reports' / 'hst' / 'capacity_internal_fusion_accepted_freezes.json'
CANDIDATE = PROJECT_ROOT / 'reports' / 'hst' / 'capacity_internal_fusion_acceptance_candidate.json'
RUN_CONFIRMATORY = False  # Set True only after independent manual promotion of CANDIDATE.
POLL_SECONDS = 60
assert CONFIG.is_file(), CONFIG
print(PROJECT_ROOT)

In [ ]:
def run_json(arguments):
    completed = subprocess.run(arguments, cwd=PROJECT_ROOT, check=True, text=True, capture_output=True)
    print(completed.stderr, end='')
    payload = json.loads(completed.stdout)
    print(json.dumps(payload, indent=2))
    return payload

def wait_for_launch(launch_id):
    while True:
        payload = run_json([sys.executable, 'scripts/72_run_hst_reliability.py', '--project-root', str(PROJECT_ROOT), '--status-id', launch_id])
        if payload['status'] in {'success', 'failed'}:
            if payload['status'] != 'success':
                raise RuntimeError(payload.get('error', 'HST launch failed'))
            return payload
        time.sleep(POLL_SECONDS)

In [ ]:
pilot = run_json([sys.executable, 'scripts/72_run_hst_reliability.py', '--config', str(CONFIG), '--project-root', str(PROJECT_ROOT), '--accepted-freezes', str(ACCEPTED), '--mode', 'pilot', '--device', 'cuda', '--through', 'base_resource_pilot', '--detach'])
pilot_status = wait_for_launch(pilot['launch_id'])
run_root = PROJECT_ROOT / 'data' / 'outputs' / 'hst' / pilot_status['run_id']
subprocess.run([sys.executable, 'scripts/75_prepare_hst_acceptance.py', '--run-root', str(run_root), '--output', str(CANDIDATE)], cwd=PROJECT_ROOT, check=True)
print(f'Review-only candidate: {CANDIDATE}')

In [ ]:
if RUN_CONFIRMATORY:
    if not ACCEPTED.is_file():
        raise FileNotFoundError('Manual approval file is absent; the confirmatory run remains blocked.')
    full = run_json([sys.executable, 'scripts/72_run_hst_reliability.py', '--config', str(CONFIG), '--project-root', str(PROJECT_ROOT), '--accepted-freezes', str(ACCEPTED), '--mode', 'full', '--device', 'cuda', '--through', 'evidence_pack', '--detach'])
    wait_for_launch(full['launch_id'])
else:
    print('Confirmatory launch not requested. Manual acceptance remains mandatory.')